# DVD (ABAW) — покадровое обнаружение насилия (frame-level)

Цель: научить модель выдавать **0/1 для каждого кадра** видео.

Идея обучения:
- из каждого видео берём **клип** длиной `T` кадров (с шагом `frame_step`)
- модель предсказывает логиты `[B, 2, T]`
- loss считается **покадрово** (masked CE), используя разметку из CSV (`Frame_Number, Label`)
- на инференсе делаем sliding-window по всему видео и усредняем вероятности по перекрытиям

⚠️ В DVD часть `.mp4` может быть закодирована **AV1**. `decord` в Colab часто падает на таких видео, поэтому здесь по умолчанию используем **PyAV** (через `av`).


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive', force_remount=True)


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")
warnings.filterwarnings("ignore", message="torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly.*", category=UserWarning,)

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:64"


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="torchvision")

In [ ]:
!pip -q install -r requirements.txt

In [ ]:
!nvidia-smi

In [ ]:
# ===== 1) Imports =====
import os, json, time, math, random
from pathlib import Path
from typing import Optional, Tuple, List, Dict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import torchvision
import torchvision.transforms.functional as TF

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None
from tqdm.auto import tqdm
from IPython.display import display

import av  # PyAV

import torch.multiprocessing as mp
try:
    mp.set_start_method("spawn", force=True)
except RuntimeError:
    pass
    

In [ ]:
# ===== 2) Config =====
CFG = dict(
    # paths
    root="/home/HDD6TB/datasets/emotions/ABAW/ABAW_9/DVD",
    train_videos="Training/videos",
    train_anns="Training/annotations",
    train_frames="Training/frames",
    val_videos="Validation/videos",
    val_anns="Validation/annotations",
    val_frames="Validation/frames",
    video_ext=".mp4",

    # data sampling
    img_size=224,
    clip_len=16,
    frame_step=2,
    clip_mode="random",
    val_clip_mode="random",
    val_n_views=7,
    val_seed=67,
    pos_sample_prob=0.0,
    train_n_views=1,
    # final training mode: include Validation into train set
    train_on_train_plus_val=False,
    # if >0 and train_on_train_plus_val=True, keep holdout from (train+val) for monitoring
    train_plus_val_holdout_ratio=0.0,

    # loader
    train_batch_size=4,
    val_batch_size=8,
    num_workers=0,

    # runtime / training loop behavior
    epochs=15,
    grad_clip=1.0,
    grad_accum_steps=4,
    amp=True,
    label_smoothing=0.05,
    loss_type="ce",      # "ce" | "focal"
    focal_gamma=2.0,
    use_tensorboard=True,

    # fail-fast modality checks (avoid silent zero fallback)
    strict_require_flow=False,
    strict_require_skeleton=False,

    # multi-GPU
    use_dataparallel=False,
    data_parallel_device_ids=[0, 1],

    # memory saver for timm encoders (convnext/efficientnet)
    encoder_grad_checkpointing=True,

    run_dir="runs",
    seed=67,

    backend="frames",  # "frames" | "pyav" | "torchvision"

    # ---- backbone choice ----
    # "resnet18" | "efficientnet_b0_bilstm" | "video_swin_tiny" | "videomae_small"
    backbone=None,

    # Single source of truth for presets:
    presets_path="skeleton_attention_boost_presets.json",
    train_preset="sk_attn_frozen_bighead_v1",  # set preset name to apply from train_presets.json
)

PRESET_OVERRIDES = {}
presets_path = Path(CFG.get("presets_path", "train_presets.json"))
if not presets_path.is_absolute():
    presets_path = Path.cwd() / presets_path

if presets_path.exists():
    PRESET_OVERRIDES = json.loads(presets_path.read_text(encoding="utf-8"))
    print(f"Loaded presets: {list(PRESET_OVERRIDES.keys())} from {presets_path}")
else:
    print(f"[warn] presets file not found: {presets_path}. Using manual CFG only.")

preset_name = CFG.get("train_preset")
if preset_name:
    if preset_name not in PRESET_OVERRIDES:
        raise ValueError(f"Unknown train_preset: {preset_name}. Available: {list(PRESET_OVERRIDES)}")
    CFG.update(PRESET_OVERRIDES[preset_name])
    print(f"Applied train preset: {preset_name}")

ROOT = Path(CFG["root"])
TRAIN_VIDEOS = ROOT / CFG["train_videos"]
TRAIN_ANNS   = ROOT / CFG["train_anns"]
TRAIN_FRAMES = ROOT / CFG["train_frames"]
VAL_VIDEOS   = ROOT / CFG["val_videos"]
VAL_ANNS     = ROOT / CFG["val_anns"]
VAL_FRAMES   = ROOT / CFG["val_frames"]

# --- Flow frames (two-stream) ---
TRAIN_FLOW_FRAMES = Path(CFG.get("flow_frames_root_train", "/flow/Training"))
VAL_FLOW_FRAMES = Path(CFG.get("flow_frames_root_val", "flow/Validation"))
USE_FLOW = bool(CFG.get("use_flow", False))
if USE_FLOW:
    print(f"[flow] TRAIN_FLOW_FRAMES={TRAIN_FLOW_FRAMES} exists={TRAIN_FLOW_FRAMES.exists()}")
    print(f"[flow] VAL_FLOW_FRAMES={VAL_FLOW_FRAMES} exists={VAL_FLOW_FRAMES.exists()}")



device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device=="cuda" else "")

if device == "cuda":
    torch.set_float32_matmul_precision("medium")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False

# --- Skeleton features ---
TRAIN_SKELETON = Path(CFG.get("skeleton_root_train", str(ROOT / "Training/skeleton")))
VAL_SKELETON = Path(CFG.get("skeleton_root_val", str(ROOT / "Validation/skeleton")))
USE_SKELETON = bool(CFG.get("use_skeleton", False))
print(f"[modality] USE_FLOW={USE_FLOW}  USE_SKELETON={USE_SKELETON}")
if USE_SKELETON:
    print("TRAIN_SKELETON:", TRAIN_SKELETON)
    print("VAL_SKELETON:", VAL_SKELETON)






In [ ]:
CFG["use_dataparallel"] = False

# CFG["data_parallel_device_ids"] = [0, 1]

In [ ]:
# ===== 3) Utils =====
def seed_everything(seed: int = 0):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG["seed"])

def seed_worker(worker_id):
    seed = CFG["seed"] + worker_id
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)

g = torch.Generator()
g.manual_seed(CFG["seed"])

def dummy_frame(size: int) -> np.ndarray:
    """HWC uint8 black frame."""
    return np.zeros((size, size, 3), dtype=np.uint8)

In [ ]:
# ===== 4) Read labels from CSV (Frame_Number, Label) =====
def load_frame_labels(csv_path: Path) -> np.ndarray:
    """
    Возвращает y_full: np.ndarray shape [N] со значениями {0,1} или -1 (если какие-то кадры отсутствуют).
    Frame_Number в CSV ожидаем 1-based.
    """
    df = pd.read_csv(csv_path)
    cols = {c.lower(): c for c in df.columns}
    fn_col = cols.get("frame_number", df.columns[0])
    lb_col = cols.get("label", df.columns[1])

    frame_num = df[fn_col].astype(int).to_numpy()
    label = df[lb_col].astype(int).to_numpy()

    n = int(frame_num.max())
    y = np.full((n,), -1, dtype=np.int64)
    idx = frame_num - 1  # 1-based -> 0-based
    ok = (idx >= 0) & (idx < n)
    y[idx[ok]] = label[ok]
    return y

def build_video_meta(ann_dir: Path) -> pd.DataFrame:
    """
    По одному ряду на видео (csv = одно видео).
    has_pos — удобный флаг для positive-aware sampling (ускоряет обучение на дисбалансе).
    """
    csvs = sorted(Path(ann_dir).glob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSVs in {ann_dir}")

    rows = []
    for p in tqdm(csvs, desc=f"meta from {ann_dir.name}"):
        vid = p.stem
        y = load_frame_labels(p)
        has_pos = int((y == 1).any())
        rows.append(dict(video=vid, csv_path=str(p), has_pos=has_pos, n_annot=int(len(y))))
    return pd.DataFrame(rows)

def split_train_val(meta: pd.DataFrame, val_ratio: float = 0.2, seed: int = 0):
    rng = np.random.RandomState(seed)
    idx = np.arange(len(meta))
    rng.shuffle(idx)
    n_val = int(len(idx) * val_ratio)
    val_idx = idx[:n_val]
    tr_idx  = idx[n_val:]
    return meta.iloc[tr_idx].reset_index(drop=True), meta.iloc[val_idx].reset_index(drop=True)

train_meta = build_video_meta(TRAIN_ANNS)
train_meta["split_src"] = "train"

if VAL_ANNS.exists() and VAL_VIDEOS.exists():
    val_meta = build_video_meta(VAL_ANNS)
    val_meta["split_src"] = "val"
else:
    train_meta, val_meta = split_train_val(train_meta, val_ratio=0.2, seed=CFG["seed"])

print("train:", train_meta.shape, "val:", val_meta.shape)
print(train_meta.head(3))

# Optional final mode: train on Training+Validation
if bool(CFG.get("train_on_train_plus_val", False)) and VAL_ANNS.exists() and VAL_VIDEOS.exists():
    _val_meta_full = build_video_meta(VAL_ANNS)
    _val_meta_full["split_src"] = "val"
    _full_meta = pd.concat([train_meta, _val_meta_full], ignore_index=True)
    _holdout_ratio = float(CFG.get("train_plus_val_holdout_ratio", 0.0))

    if _holdout_ratio > 0.0:
        train_meta, val_meta = split_train_val(_full_meta, val_ratio=_holdout_ratio, seed=CFG["seed"])
        print(f"[train+val] ON with holdout_ratio={_holdout_ratio:.3f}: train={train_meta.shape}, val={val_meta.shape}")
    else:
        train_meta = _full_meta.reset_index(drop=True)
        print("[train+val] ON without holdout: train uses Training+Validation; val remains previous split (leaky monitor)")

    print("train:", train_meta.shape, "val:", val_meta.shape)


In [ ]:
# ===== 5) Video decoding (PyAV / torchvision / frames) =====
def _read_clip_rgb_pyav(video_path: Path, start: int, clip_len: int, frame_step: int) -> Optional[np.ndarray]:
    try:
        container = av.open(str(video_path))
        stream = next(s for s in container.streams if s.type == "video")
        fps = float(stream.average_rate) if stream.average_rate is not None else 30.0
        tb = float(stream.time_base)

        idxs = [int(start + k * frame_step) for k in range(clip_len)]
        want = set(idxs)
        first, last = idxs[0], idxs[-1]

        seek_time = max(0.0, first / fps)
        seek_pts = int(seek_time / tb) if tb > 0 else 0
        try:
            container.seek(seek_pts, stream=stream)
        except Exception:
            pass

        out = {}
        for frame in container.decode(stream):
            if frame.pts is None:
                continue
            fi = int(round(frame.pts * tb * fps))
            if fi < first - 2:
                continue
            if fi > last + 2:
                break
            if fi in want:
                out[fi] = frame.to_rgb().to_ndarray()
                if len(out) == len(want):
                    break

        container.close()
        if len(out) == 0:
            return None

        keys_sorted = sorted(out.keys())
        fallback = out[keys_sorted[0]]
        clip = []
        for fi in idxs:
            if fi in out:
                clip.append(out[fi])
            else:
                nearest = min(keys_sorted, key=lambda k: abs(k - fi))
                clip.append(out.get(nearest, fallback))
        return np.stack(clip, axis=0)
    except Exception:
        return None


def _read_clip_rgb_torchvision(video_path: Path, start: int, clip_len: int, frame_step: int) -> Optional[np.ndarray]:
    try:
        vr = torchvision.io.VideoReader(str(video_path), "video")
        meta = vr.get_metadata()
        fps = float(meta["video"]["fps"][0]) if "video" in meta and "fps" in meta["video"] else 30.0

        idxs = [int(start + k * frame_step) for k in range(clip_len)]
        out = []
        for fi in idxs:
            t = fi / fps
            vr.seek(t)
            fr = next(vr)
            x = fr["data"]
            if x.ndim == 3 and x.shape[-1] >= 3:
                x = x[..., :3]
            out.append(x.cpu().numpy())
        return np.stack(out, axis=0)
    except Exception:
        return None


def _read_frame_rgb(img_path: Path) -> Optional[np.ndarray]:
    try:
        x = torchvision.io.read_image(str(img_path))
        if x.ndim != 3:
            return None
        if x.shape[0] == 1:
            x = x.repeat(3, 1, 1)
        elif x.shape[0] >= 3:
            x = x[:3]
        return x.permute(1, 2, 0).contiguous().cpu().numpy()
    except Exception:
        return None


def _build_frame_index(frames_dir: Path) -> Optional[Tuple[List[int], Dict[int, Path]]]:
    if frames_dir is None or (not frames_dir.exists()):
        return None

    fmap: Dict[int, Path] = {}
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        for p in frames_dir.glob(ext):
            try:
                k = int(p.stem)
            except Exception:
                continue
            fmap[k] = p

    if not fmap:
        return None

    keys = sorted(fmap.keys())
    return keys, fmap


def _read_clip_rgb_frames(
    frames_dir: Path,
    start: int,
    clip_len: int,
    frame_step: int,
    frame_index: Optional[Tuple[List[int], Dict[int, Path]]] = None,
) -> Optional[np.ndarray]:
    if frame_index is None:
        frame_index = _build_frame_index(frames_dir)
    if frame_index is None:
        return None

    keys, fmap = frame_index
    if len(keys) == 0:
        return None

    idxs_1based = [int(start + k * frame_step + 1) for k in range(clip_len)]
    out = []
    first_ok = None

    for fi in idxs_1based:
        p = fmap.get(fi)
        if p is None:
            nearest = min(keys, key=lambda k: abs(k - fi))
            p = fmap.get(nearest)
        fr = _read_frame_rgb(p) if p is not None else None
        if fr is None:
            if first_ok is not None:
                fr = first_ok.copy()
            else:
                return None
        if first_ok is None:
            first_ok = fr
        out.append(fr)

    return np.stack(out, axis=0) if out else None


def read_clip_rgb(
    video_path: Optional[Path],
    start: int,
    clip_len: int,
    frame_step: int,
    backend: str = "pyav",
    frames_dir: Optional[Path] = None,
    frame_index: Optional[Tuple[List[int], Dict[int, Path]]] = None,
) -> Optional[np.ndarray]:
    if backend == "frames":
        return _read_clip_rgb_frames(
            frames_dir=frames_dir,
            start=start,
            clip_len=clip_len,
            frame_step=frame_step,
            frame_index=frame_index,
        )

    if video_path is None:
        return None

    if backend == "torchvision":
        rgb = _read_clip_rgb_torchvision(video_path, start, clip_len, frame_step)
        if rgb is not None:
            return rgb
        return _read_clip_rgb_pyav(video_path, start, clip_len, frame_step)

    rgb = _read_clip_rgb_pyav(video_path, start, clip_len, frame_step)
    if rgb is not None:
        return rgb
    return _read_clip_rgb_torchvision(video_path, start, clip_len, frame_step)



In [ ]:
# ===== 6) Torch-only transforms (без PIL) =====
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

class TorchVideoTransform:
    """
    Вход кадра: torch.Tensor [C,H,W] (uint8 0..255 или float)
    Выход: float32 [C,H,W], resized + normalized.
    """
    def __init__(self, size: int, train: bool):
        self.size = int(size)
        self.train = bool(train)

    def __call__(self, x: torch.Tensor) -> torch.Tensor:
        if not isinstance(x, torch.Tensor):
            x = torch.from_numpy(x)

        # HWC -> CHW
        if x.ndim == 3 and x.shape[-1] in (1,3,4) and x.shape[0] not in (1,3,4):
            x = x.permute(2,0,1)

        if x.shape[0] == 4:
            x = x[:3]

        if x.dtype == torch.uint8:
            x = x.float().div_(255.0)
        else:
            x = x.float()
            if x.max() > 1.5:
                x = x / 255.0

        x = TF.resize(x, [self.size, self.size], antialias=True)

        if self.train and torch.rand(()) < 0.5:
            x = TF.hflip(x)

        x = TF.normalize(x, mean=IMAGENET_MEAN, std=IMAGENET_STD)
        return x


# Baseline transforms
train_tfms = TorchVideoTransform(CFG["img_size"], train=True)
val_tfms   = TorchVideoTransform(CFG["img_size"], train=False)


# Optional: TrivialAugmentWide (train only). This matches the training patch used in best runs.
if bool(CFG.get("use_trivial_aug_wide", False)):
    try:
        import torchvision.transforms as T

        class TorchVideoTransformTrivialWide:
            def __init__(self, size: int, train: bool):
                self.size = int(size)
                self.train = bool(train)
                bins = int(CFG.get("trivial_aug_num_magnitude_bins", 31))
                self.aug = T.TrivialAugmentWide(num_magnitude_bins=bins)

            def _to_chw_float(self, x: torch.Tensor) -> torch.Tensor:
                if not isinstance(x, torch.Tensor):
                    x = torch.from_numpy(x)
                if x.ndim == 3 and x.shape[-1] in (1, 3, 4) and x.shape[0] not in (1, 3, 4):
                    x = x.permute(2, 0, 1)
                if x.shape[0] == 4:
                    x = x[:3]
                if x.dtype == torch.uint8:
                    x = x.float().div_(255.0)
                else:
                    x = x.float()
                    if x.max() > 1.5:
                        x = x / 255.0
                return x

            def __call__(self, x: torch.Tensor) -> torch.Tensor:
                x = self._to_chw_float(x).clamp_(0.0, 1.0)
                x = TF.resize(x, [self.size, self.size], antialias=True)
                if self.train:
                    x_pil = TF.to_pil_image((x * 255.0).to(torch.uint8))
                    x_pil = self.aug(x_pil)
                    x = TF.pil_to_tensor(x_pil).float().div_(255.0)
                x = TF.normalize(x, mean=IMAGENET_MEAN, std=IMAGENET_STD)
                return x

        train_tfms = TorchVideoTransformTrivialWide(CFG["img_size"], train=True)
        val_tfms   = TorchVideoTransformTrivialWide(CFG["img_size"], train=False)
        print("[tfms] TrivialAugmentWide enabled (train only)")
    except Exception as e:
        print(f"[tfms][warn] TrivialAugmentWide patch failed: {e}")



In [ ]:
# ===== 7) Dataset (clip -> per-frame labels) =====
class DVDClipFrameLabelDataset(Dataset):
    def __init__(
        self,
        meta: pd.DataFrame,
        videos_root: Optional[Path],
        transform,
        video_ext: str = ".mp4",
        clip_len: int = 16,
        frame_step: int = 2,
        clip_mode: str = "random",
        train: bool = True,
        img_size: int = 224,
        pos_sample_prob: float = 0.7,
        backend: str = "pyav",
        seed: int = 67,
        deterministic: bool = False,
        id_col: str = "video",
        n_views: int = 1,
        frames_root: Optional[Path] = None,
        flow_frames_root: Optional[Path] = None,
        skeleton_root: Optional[Path] = None,
    ):
        self.meta = meta.reset_index(drop=True)
        self.videos_root = Path(videos_root) if videos_root is not None else None
        self.frames_root = Path(frames_root) if frames_root is not None else None
        self.flow_frames_root = Path(flow_frames_root) if flow_frames_root is not None else None
        self.skeleton_root = Path(skeleton_root) if skeleton_root is not None else None
        self.transform = transform
        self.video_ext = video_ext
        self.clip_len = int(clip_len)
        self.frame_step = int(frame_step)
        self.clip_mode = clip_mode
        self.train = bool(train)
        self.img_size = int(img_size)
        self.pos_sample_prob = float(pos_sample_prob)
        self.backend = str(backend)
        self.span = (self.clip_len - 1) * self.frame_step + 1

        self.seed = int(seed)
        self.deterministic = bool(deterministic)
        self.id_col = str(id_col)
        self.n_views = int(n_views)

        _cfg = globals().get("CFG", {})
        self.train_n_views = max(1, int(_cfg.get("train_n_views", 1)))
        self.strict_require_flow = bool(_cfg.get("strict_require_flow", False))
        self.strict_require_skeleton = bool(_cfg.get("strict_require_skeleton", False))

        self._frame_index_cache: Dict[str, Optional[Tuple[List[int], Dict[int, Path]]]] = {}

    def __len__(self):
        if self.train:
            return len(self.meta) * self.train_n_views
        return len(self.meta) * self.n_views

    def _stable_seed(self, idx: int, row, view: int = 0) -> int:
        s = f"{row.get(self.id_col, idx)}_{view}"
        h = 0
        for ch in s:
            h = (h * 131 + ord(ch)) % (2**31 - 1)
        return (self.seed + h) % (2**31 - 1)

    def _choose_start(self, y: np.ndarray, has_pos: int, idx: int, row, view: int = 0) -> int:
        n_frames = len(y)
        if n_frames <= self.span:
            return 0

        if self.clip_mode == "center":
            return int((n_frames - self.span) // 2)

        if (not self.train) and self.clip_mode == "random":
            if self.deterministic:
                rs = np.random.RandomState(self._stable_seed(idx, row, view=view))
                return int(rs.randint(0, n_frames - self.span + 1))
            return int(np.random.randint(0, n_frames - self.span + 1))

        if has_pos == 1 and np.random.rand() < self.pos_sample_prob:
            pos_idx = np.where(y == 1)[0]
            if len(pos_idx) > 0:
                center = int(np.random.choice(pos_idx))
                start = center - (self.span // 2)
                return int(np.clip(start, 0, n_frames - self.span))

        return int(np.random.randint(0, n_frames - self.span + 1))

    def _get_frame_index(self, vid: str) -> Optional[Tuple[List[int], Dict[int, Path]]]:
        if vid in self._frame_index_cache:
            return self._frame_index_cache[vid]

        if self.frames_root is None:
            self._frame_index_cache[vid] = None
            return None

        idx = _build_frame_index(self.frames_root / vid)
        self._frame_index_cache[vid] = idx
        return idx

    def __getitem__(self, idx: int):
        idx = int(idx)

        if self.train and getattr(self, "train_n_views", 1) > 1:
            n_views = int(self.train_n_views)
            vid_idx = idx // n_views
            view = idx % n_views
        elif (not self.train) and getattr(self, "n_views", 1) > 1:
            n_views = int(self.n_views)
            vid_idx = idx // n_views
            view = idx % n_views
        else:
            vid_idx = idx
            view = 0

        row = self.meta.iloc[vid_idx]
        vid = row["video"]
        csv_path = Path(row["csv_path"])
        has_pos = int(row.get("has_pos", 0))

        y = load_frame_labels(csv_path)
        start = self._choose_start(y, has_pos, idx=vid_idx, row=row, view=view)

        frame_index = None
        frames_dir = None
        video_path = None

        row_split = str(row.get("split_src", "train")).lower()
        train_plus_val_mode = bool(globals().get("CFG", {}).get("train_on_train_plus_val", False))

        if self.backend == "frames":
            chosen_root = self.frames_root
            if train_plus_val_mode:
                if row_split == "val" and "VAL_FRAMES" in globals() and Path(globals()["VAL_FRAMES"]).exists():
                    chosen_root = Path(globals()["VAL_FRAMES"])
                elif row_split == "train" and "TRAIN_FRAMES" in globals() and Path(globals()["TRAIN_FRAMES"]).exists():
                    chosen_root = Path(globals()["TRAIN_FRAMES"])

            frames_dir = chosen_root / vid if chosen_root is not None else None
            if frames_dir is not None:
                _k = f"_rgb_{frames_dir}"
                if _k in self._frame_index_cache:
                    frame_index = self._frame_index_cache[_k]
                else:
                    frame_index = _build_frame_index(frames_dir)
                    self._frame_index_cache[_k] = frame_index
        elif self.videos_root is not None:
            chosen_vroot = self.videos_root
            if train_plus_val_mode:
                if row_split == "val" and "VAL_VIDEOS" in globals() and Path(globals()["VAL_VIDEOS"]).exists():
                    chosen_vroot = Path(globals()["VAL_VIDEOS"])
                elif row_split == "train" and "TRAIN_VIDEOS" in globals() and Path(globals()["TRAIN_VIDEOS"]).exists():
                    chosen_vroot = Path(globals()["TRAIN_VIDEOS"])
            video_path = chosen_vroot / f"{vid}{self.video_ext}" if chosen_vroot is not None else None

        rgb = read_clip_rgb(
            video_path,
            start=start,
            clip_len=self.clip_len,
            frame_step=self.frame_step,
            backend=self.backend,
            frames_dir=frames_dir,
            frame_index=frame_index,
        )

        if rgb is None or rgb.shape[0] != self.clip_len:
            rgb = np.stack([dummy_frame(self.img_size)] * self.clip_len, axis=0)

        idxs = start + np.arange(self.clip_len) * self.frame_step
        idxs = np.clip(idxs, 0, len(y) - 1).astype(int)

        y_seq = y[idxs].astype(np.int64)
        mask = (y_seq != -1)

        frames = []
        for t in range(self.clip_len):
            fr = rgb[t]
            fr_t = torch.from_numpy(fr).permute(2, 0, 1)
            frames.append(self.transform(fr_t))

        clip = torch.stack(frames, dim=0).permute(1, 0, 2, 3)

        extra = None

        # --- Flow (two-stream) ---
        if self.flow_frames_root is not None:
            flow_root = self.flow_frames_root
            if train_plus_val_mode:
                if row_split == "val" and "VAL_FLOW_FRAMES" in globals() and Path(globals()["VAL_FLOW_FRAMES"]).exists():
                    flow_root = Path(globals()["VAL_FLOW_FRAMES"])
                elif row_split == "train" and "TRAIN_FLOW_FRAMES" in globals() and Path(globals()["TRAIN_FLOW_FRAMES"]).exists():
                    flow_root = Path(globals()["TRAIN_FLOW_FRAMES"])

            flow_dir = flow_root / vid if flow_root is not None else None
            if self.strict_require_flow and (flow_dir is None or not flow_dir.exists()):
                raise FileNotFoundError(f"Flow dir not found for video={vid}: {flow_dir}")

            _fkey = f"_flow_{vid}"
            if _fkey in self._frame_index_cache:
                flow_frame_index = self._frame_index_cache[_fkey]
            else:
                flow_frame_index = _build_frame_index(flow_dir) if flow_dir.exists() else None
                self._frame_index_cache[_fkey] = flow_frame_index

            flow_rgb = read_clip_rgb(
                video_path=None,
                start=start,
                clip_len=self.clip_len,
                frame_step=self.frame_step,
                backend="frames",
                frames_dir=flow_dir,
                frame_index=flow_frame_index,
            )

            if flow_rgb is None or flow_rgb.shape[0] != self.clip_len:
                if self.strict_require_flow:
                    raise RuntimeError(
                        f"Flow clip load failed for video={vid}, start={start}, "
                        f"clip_len={self.clip_len}, frame_step={self.frame_step}"
                    )
                flow_rgb = np.stack([dummy_frame(self.img_size)] * self.clip_len, axis=0)

            flow_frames = []
            for t in range(self.clip_len):
                fr = flow_rgb[t]
                fr_t = torch.from_numpy(fr).permute(2, 0, 1)
                flow_frames.append(self.transform(fr_t))

            extra = torch.stack(flow_frames, dim=0).permute(1, 0, 2, 3)

        # --- Skeleton (two-stream / extra modality) ---
        elif self.skeleton_root is not None:
            skel_root = self.skeleton_root
            if train_plus_val_mode:
                if row_split == "val" and "VAL_SKELETON" in globals() and Path(globals()["VAL_SKELETON"]).exists():
                    skel_root = Path(globals()["VAL_SKELETON"])
                elif row_split == "train" and "TRAIN_SKELETON" in globals() and Path(globals()["TRAIN_SKELETON"]).exists():
                    skel_root = Path(globals()["TRAIN_SKELETON"])

            skel_path = skel_root / f"{vid}.npy"

            if skel_path.exists():
                try:
                    skel = np.load(skel_path)
                except Exception as e:
                    if self.strict_require_skeleton:
                        raise RuntimeError(f"Skeleton load failed for {skel_path}: {e}")
                    skel = np.zeros((len(y), 406), dtype=np.float32)
            else:
                if self.strict_require_skeleton:
                    raise FileNotFoundError(f"Skeleton file not found: {skel_path}")
                skel = np.zeros((len(y), 406), dtype=np.float32)

            if skel.ndim != 2:
                if self.strict_require_skeleton:
                    raise RuntimeError(f"Skeleton must be 2D [N,D], got shape={getattr(skel, 'shape', None)} for {skel_path}")
                skel = np.zeros((len(y), 406), dtype=np.float32)

            feat_dim = int(skel.shape[1]) if skel.shape[1] > 0 else 406

            if len(skel) == 0:
                if self.strict_require_skeleton:
                    raise RuntimeError(f"Skeleton has zero frames for video={vid} ({skel_path})")
                skel_clip = np.zeros((self.clip_len, feat_dim), dtype=np.float32)
            else:
                safe_idxs = np.clip(idxs, 0, len(skel) - 1).astype(int)
                skel_clip = skel[safe_idxs].astype(np.float32)

            if skel_clip.shape[0] != self.clip_len:
                pad = np.zeros((self.clip_len, feat_dim), dtype=np.float32)
                take = min(self.clip_len, skel_clip.shape[0])
                pad[:take] = skel_clip[:take]
                skel_clip = pad

            extra = torch.from_numpy(skel_clip).float()

        if extra is not None:
            return clip, torch.from_numpy(y_seq), torch.from_numpy(mask), extra

        return clip, torch.from_numpy(y_seq), torch.from_numpy(mask)

        


In [ ]:
# ===== 8) DataLoaders =====
train_ds = DVDClipFrameLabelDataset(
    meta=train_meta,
    videos_root=TRAIN_VIDEOS,
    frames_root=TRAIN_FRAMES,
    flow_frames_root=TRAIN_FLOW_FRAMES if USE_FLOW else None,
    skeleton_root=TRAIN_SKELETON if USE_SKELETON else None,
    transform=train_tfms,
    video_ext=CFG["video_ext"],
    clip_len=CFG["clip_len"],
    frame_step=CFG["frame_step"],
    clip_mode="random",
    train=True,
    img_size=CFG["img_size"],
    pos_sample_prob=CFG["pos_sample_prob"],
    backend=CFG["backend"],
    seed=CFG["seed"],
    deterministic=False,
    id_col="video",
)

val_ds = DVDClipFrameLabelDataset(
    meta=val_meta,
    videos_root=VAL_VIDEOS if VAL_VIDEOS.exists() else TRAIN_VIDEOS,
    frames_root=VAL_FRAMES if VAL_FRAMES.exists() else TRAIN_FRAMES,
    flow_frames_root=VAL_FLOW_FRAMES if (USE_FLOW and VAL_FLOW_FRAMES.exists()) else None,
    skeleton_root=VAL_SKELETON if USE_SKELETON else None,
    transform=val_tfms,
    video_ext=CFG["video_ext"],
    clip_len=CFG["clip_len"],
    frame_step=CFG["frame_step"],
    clip_mode=CFG.get("val_clip_mode", "random"),
    train=False,
    img_size=CFG["img_size"],
    pos_sample_prob=0.0,
    backend=CFG["backend"],
    seed=CFG.get("val_seed", CFG["seed"]),
    deterministic=True,
    id_col="video",
    n_views=CFG.get("val_n_views", 7),
)

def make_loader(ds, shuffle: bool, batch_size: Optional[int] = None):
    nw = CFG["num_workers"]
    return DataLoader(
        ds,
        batch_size=batch_size if batch_size is not None else CFG["train_batch_size"],
        shuffle=shuffle,
        num_workers=nw,
        pin_memory=(device=="cuda"),
        persistent_workers=False if nw > 0 else False,
        prefetch_factor=1 if nw > 0 else None,
        drop_last=shuffle,
        worker_init_fn=seed_worker if nw > 0 else None,
        generator=g if shuffle else None,
    )

train_loader = make_loader(train_ds, shuffle=True, batch_size=CFG["train_batch_size"])
val_loader   = make_loader(val_ds, shuffle=False, batch_size=CFG["val_batch_size"])

print("len(train_loader), len(val_loader):", len(train_loader), len(val_loader))

t0 = time.time()
_batch = next(iter(train_loader))
x, y_seq, mask = _batch[0], _batch[1], _batch[2]

if len(_batch) == 4:
    if USE_SKELETON:
        print(f"[skeleton] skeleton shape: {tuple(_batch[3].shape)}")
    elif USE_FLOW:
        print(f"[flow] flow_clip shape: {tuple(_batch[3].shape)}")
print("first batch sec:", time.time()-t0)
print("x:", tuple(x.shape), "y_seq:", tuple(y_seq.shape), "mask:", tuple(mask.shape), "mask mean:", float(mask.float().mean()))
print("TRAIN backend:", train_ds.backend, "frames_root:", train_ds.frames_root, "videos_root:", train_ds.videos_root)
print("VAL backend:", val_ds.backend, "frames_root:", val_ds.frames_root, "videos_root:", val_ds.videos_root)
print("TRAIN_FRAMES exists:", TRAIN_FRAMES.exists(), "VAL_FRAMES exists:", VAL_FRAMES.exists())



In [ ]:
sample = train_ds[0]
print("len(sample) =", len(sample))
for i, x in enumerate(sample):
    if torch.is_tensor(x):
        print(i, tuple(x.shape), x.dtype)
    else:
        print(i, type(x))

In [ ]:
_s = val_ds[0]
x1, y1, m1 = _s[0], _s[1], _s[2]
_s2 = val_ds[0]
x2, y2, m2 = _s2[0], _s2[1], _s2[2]
print("same labels?", torch.equal(y1, y2), "same mask?", torch.equal(m1, m2))

In [ ]:
_s = val_ds[0]
x1, y1, m1 = _s[0], _s[1], _s[2]
_s2 = val_ds[0]
x2, y2, m2 = _s2[0], _s2[1], _s2[2]
print("same x?", torch.allclose(x1, x2))
print("max abs diff:", (x1 - x2).abs().max().item())

## Backbone Options

Переключай backbone через `CFG["backbone"]` в ячейке Config:

| backbone | Параметры | Архитектура | Pretrain | Рекомендуется |
|---|---|---|---|---|
| `resnet18` | 11M | Frame-level CNN + 1x1 conv | ImageNet | Baseline |
| `efficientnet_b0_bilstm` | 18M | Frame-level CNN + BiLSTM | ImageNet | ↑ над baseline |
| `video_swin_tiny` | 28M | Spatiotemporal Transformer | Kinetics-400 | **Рекомендуется** |
| `videomae_small` | 22M | MAE Video Transformer | Kinetics-400 | **Рекомендуется** |

**Важно про batch_size**: Video Swin и VideoMAE требуют больше GPU памяти.
- Colab T4 (~15GB): batch_size=4, grad_accum_steps=4
- Вышкинский сервер: batch_size=8, grad_accum_steps=2

**Порядок экспериментов**:
1. Сначала проверь `video_swin_tiny` — быстро грузится, стабильно обучается
2. Потом `videomae_small` — чуть медленнее, но потенциально выше качество
3. `efficientnet_b0_bilstm` — хороший промежуточный вариант если памяти мало


In [ ]:
# ===== 9) Models =====
# Все модели вынесены в models.py — один файл, один build_model().
# Бэкбоны: resnet18, efficientnet_b0_bilstm, video_swin_tiny, videomae_small,
#           videomae_crime_init, slowfast_r50, r3d_18_temporal, r2plus1d_18_temporal,
#           s3d_temporal, + V2 версии с BiLSTM head (*_v2).

import sys
from pathlib import Path

# Добавляем директорию с models.py в path (на случай если лежит рядом с ноутбуком)
_models_dir = str(Path.cwd())
if _models_dir not in sys.path:
    sys.path.insert(0, _models_dir)

from models import build_model, ALL_BACKBONES

print(f"Loaded models.py — {len(ALL_BACKBONES)} backbones available")
print(f"Backbones: {', '.join(ALL_BACKBONES)}")


In [ ]:
# ===== 10) Loss + metrics =====
def masked_ce_loss(
    logits: torch.Tensor,
    y_seq: torch.Tensor,
    mask: torch.Tensor,
    class_weights: Optional[torch.Tensor] = None,
    label_smoothing: float = 0.0,
    loss_type: str = "ce",
    focal_gamma: float = 2.0,
) -> torch.Tensor:
    B, K, T = logits.shape
    logits2 = logits.permute(0, 2, 1).reshape(B * T, K)
    y2 = y_seq.reshape(B * T)
    m2 = mask.reshape(B * T)

    logits2 = logits2[m2]
    y2 = y2[m2].long()
    if logits2.numel() == 0:
        return (logits * 0.0).sum()  # сохраняет grad_fn

    loss_type = str(loss_type).lower()
    if loss_type in {"ce", "cross_entropy"}:
        ce = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=label_smoothing)
        return ce(logits2, y2)

    if loss_type == "focal":
        # Multiclass focal loss over valid frame logits.
        logp = torch.log_softmax(logits2, dim=1)
        logpt = logp.gather(1, y2.unsqueeze(1)).squeeze(1)
        pt = logpt.exp()

        if class_weights is not None:
            alpha_t = class_weights[y2]
        else:
            alpha_t = torch.ones_like(pt)

        gamma = float(focal_gamma)
        focal_factor = (1.0 - pt).pow(gamma)
        loss = -alpha_t * focal_factor * logpt
        return loss.mean()

    raise ValueError(f"Unknown loss_type: {loss_type!r}. Use 'ce' or 'focal'.")

def per_class_prf1_torch(y_true: torch.Tensor, y_pred: torch.Tensor, num_classes: int = 2):
    prec, rec, f1, support = [], [], [], []
    for c in range(num_classes):
        tp = ((y_true == c) & (y_pred == c)).sum().item()
        fp = ((y_true != c) & (y_pred == c)).sum().item()
        fn = ((y_true == c) & (y_pred != c)).sum().item()

        p = tp / (tp + fp + 1e-9)
        r = tp / (tp + fn + 1e-9)
        f = 2 * p * r / (p + r + 1e-9)

        prec.append(float(p))
        rec.append(float(r))
        f1.append(float(f))
        support.append(int((y_true == c).sum().item()))

    return {"prec": prec, "rec": rec, "f1": f1, "support": support}

def macro_f1_torch(y_true: torch.Tensor, y_pred: torch.Tensor, num_classes: int = 2) -> float:
    f1s = []
    for c in range(num_classes):
        tp = ((y_true == c) & (y_pred == c)).sum().item()
        fp = ((y_true != c) & (y_pred == c)).sum().item()
        fn = ((y_true == c) & (y_pred != c)).sum().item()
        prec = tp / (tp + fp + 1e-9)
        rec  = tp / (tp + fn + 1e-9)
        f1 = 2 * prec * rec / (prec + rec + 1e-9)
        f1s.append(f1)
    return float(sum(f1s) / len(f1s))

def compute_frame_class_weights_from_csv(
    train_meta,
    device,
    load_frame_labels_fn,
    csv_col: str = "csv_path",
    ignore_label: int = -1,
    extra_pos_mult: float = 1.0,
    base_dir: Optional[str] = None,
):
    cnt0, cnt1 = 0, 0

    for p in train_meta[csv_col].tolist():
        path = Path(p)
        if base_dir is not None:
            path = Path(base_dir) / path

        y = load_frame_labels_fn(path)      # -1/0/1
        y = y[y != ignore_label]
        cnt0 += int((y == 0).sum())
        cnt1 += int((y == 1).sum())

    cnt0 = max(cnt0, 1)
    cnt1 = max(cnt1, 1)

    total = cnt0 + cnt1
    w0 = total / (2.0 * cnt0)
    w1 = total / (2.0 * cnt1) * float(extra_pos_mult)

    class_weights = torch.tensor([w0, w1], dtype=torch.float32, device=device)
    return class_weights, (cnt0, cnt1)


class_weights, (tr0, tr1) = compute_frame_class_weights_from_csv(
    train_meta=train_meta,
    device=device,
    load_frame_labels_fn=load_frame_labels,
    csv_col="csv_path",
    ignore_label=-1,
    extra_pos_mult=float(CFG.get("class_weight_pos_mult", 1.0)),
    base_dir=None,
)

print("Train frame counts:", tr0, tr1)
print("Train shares:", tr0/(tr0+tr1), tr1/(tr0+tr1))
print("Train class_weights:", class_weights.tolist())

val_class_weights, (v0, v1) = compute_frame_class_weights_from_csv(
    train_meta=val_meta,              # <-- вот это меняем
    device=device,
    load_frame_labels_fn=load_frame_labels,
    csv_col="csv_path",
    ignore_label=-1,
    extra_pos_mult=float(CFG.get("class_weight_pos_mult", 1.0)),
    base_dir=None,
)

print("VAL frame counts:", v0, v1)
print("VAL shares:", v0/(v0+v1), v1/(v0+v1))


In [ ]:
# ===== 11) Train / validation loops =====
def training_epoch(
    model,
    loader,
    optimizer,
    device,
    scaler=None,
    grad_clip: float = 0.0,
    grad_accum_steps: int = 1,
    class_weights=None,
    label_smoothing: float = 0.0,
    loss_type: str = "ce",
    focal_gamma: float = 2.0,
    scheduler=None,                  # NEW
    scheduler_step: str = "epoch",   # "iter"|"epoch"|"plateau"
    boundary_loss_fn=None,
):
    model.train()
    total_loss, n_items = 0.0, 0

    use_flow = bool(globals().get("CFG", {}).get("use_flow", False))
    use_skeleton = bool(globals().get("CFG", {}).get("use_skeleton", False))

    pbar = tqdm(loader, desc="train", dynamic_ncols=True)
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(pbar):
        if len(batch) == 4:
            x, y_seq, mask, extra = batch
            extra = extra.to(device, non_blocking=True)
        else:
            x, y_seq, mask = batch
            extra = None

        flow = extra if use_flow else None
        skeleton = extra if use_skeleton else None

        x = x.to(device, non_blocking=True)
        y_seq = y_seq.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=(scaler is not None)):
            if flow is not None:
                logits = model(x, flow=flow)
            elif skeleton is not None:
                logits = model(x, skeleton=skeleton)
            else:
                logits = model(x)

            loss = masked_ce_loss(
                logits, y_seq, mask,
                class_weights=class_weights,
                label_smoothing=label_smoothing,
                loss_type=loss_type,
                focal_gamma=focal_gamma,
            )
            if boundary_loss_fn is not None:
                b_loss = boundary_loss_fn(logits, y_seq, mask)
                loss = loss + 0.5 * b_loss
            loss = loss / grad_accum_steps

        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (step + 1) % grad_accum_steps == 0:
            if grad_clip and grad_clip > 0:
                if scaler is not None:
                    scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)
            if scheduler is not None and scheduler_step == "iter":
                scheduler.step()

        bs = x.size(0)
        total_loss += float(loss.detach().cpu()) * bs * grad_accum_steps
        n_items += bs

        lr_now = optimizer.param_groups[0]["lr"]
        pbar.set_postfix(loss=float(loss.detach().cpu()) * grad_accum_steps, lr=lr_now)

    lr_now = optimizer.param_groups[0]["lr"]
    return {"loss": total_loss / max(n_items, 1), "lr": lr_now}


@torch.no_grad()
def validation_epoch(
    model,
    loader,
    device,
    num_classes: int = 2,
    class_weights=None,
    label_smoothing: float = 0.0,
    loss_type: str = "ce",
    focal_gamma: float = 2.0,
    verbose: bool = True,
):
    model.eval()

    _cfg = globals().get("CFG", {})
    use_val_amp = bool(device == "cuda" and _cfg.get("val_amp", _cfg.get("amp", True)))
    val_micro_bs = int(_cfg.get("val_micro_batch_size", 0) or 0)

    use_flow = bool(_cfg.get("use_flow", False))
    use_skeleton = bool(_cfg.get("use_skeleton", False))

    total_loss = 0.0
    n_valid_total = 0
    all_y, all_p = [], []

    for batch in tqdm(loader, desc="val", dynamic_ncols=True):
        if len(batch) == 4:
            x, y_seq, mask, extra = batch
            extra = extra.to(device, non_blocking=True)
        else:
            x, y_seq, mask = batch
            extra = None

        flow = extra if use_flow else None
        skeleton = extra if use_skeleton else None

        x = x.to(device, non_blocking=True)
        y_seq = y_seq.to(device, non_blocking=True)
        mask = mask.to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda", enabled=use_val_amp):
            if val_micro_bs > 0 and x.size(0) > val_micro_bs:
                logits_parts = []
                for s in range(0, x.size(0), val_micro_bs):
                    xe = x[s:s + val_micro_bs]
                    fe = flow[s:s + val_micro_bs] if flow is not None else None
                    se = skeleton[s:s + val_micro_bs] if skeleton is not None else None

                    if fe is not None:
                        lp = model(xe, flow=fe)
                    elif se is not None:
                        lp = model(xe, skeleton=se)
                    else:
                        lp = model(xe)
                    logits_parts.append(lp)
                logits = torch.cat(logits_parts, dim=0)
            else:
                if flow is not None:
                    logits = model(x, flow=flow)
                elif skeleton is not None:
                    logits = model(x, skeleton=skeleton)
                else:
                    logits = model(x)

            loss = masked_ce_loss(
                logits, y_seq, mask,
                class_weights=class_weights,
                label_smoothing=label_smoothing,
                loss_type=loss_type,
                focal_gamma=focal_gamma,
            )

        pred = torch.argmax(logits, dim=1)

        yt = y_seq[mask].detach().cpu().long()
        yp = pred[mask].detach().cpu().long()

        all_y.append(yt)
        all_p.append(yp)

        n_valid = int(mask.sum().item())
        total_loss += float(loss.detach().cpu()) * max(n_valid, 1)
        n_valid_total += n_valid

    if len(all_y) == 0:
        return {
            "loss": total_loss / max(n_valid_total, 1),
            "macro_f1": 0.0,
            "pred_dist": [0.0] * num_classes,
            "true_dist": [0.0] * num_classes,
        }

    y_true = torch.cat(all_y, dim=0)
    y_pred = torch.cat(all_p, dim=0)

    true_counts = torch.bincount(y_true, minlength=num_classes)
    pred_counts = torch.bincount(y_pred, minlength=num_classes)

    n_total = int(true_counts.sum().item())
    n0 = int(true_counts[0].item())
    n1 = int(true_counts[1].item())

    tn = int(((y_true == 0) & (y_pred == 0)).sum().item())
    fp = int(((y_true == 0) & (y_pred == 1)).sum().item())
    fn = int(((y_true == 1) & (y_pred == 0)).sum().item())
    tp = int(((y_true == 1) & (y_pred == 1)).sum().item())

    if verbose:
        print(f"pred1/total: {int(pred_counts[1].item())} / {n_total}  true1/total: {n1} / {n_total}")
        print("TN FP FN TP:", tn, fp, fn, tp)

    mf1 = macro_f1_torch(y_true, y_pred, num_classes=num_classes)
    pc = per_class_prf1_torch(y_true, y_pred, num_classes=num_classes)

    pred_dist = (pred_counts.float() / pred_counts.sum().clamp_min(1)).tolist()
    true_dist = (true_counts.float() / true_counts.sum().clamp_min(1)).tolist()

    return {
        "loss": total_loss / max(n_valid_total, 1),
        "macro_f1": float(mf1),
        "pred_dist": pred_dist,
        "true_dist": true_dist,
        "prec_per_class": pc["prec"],
        "rec_per_class": pc["rec"],
        "f1_per_class": pc["f1"],
        "support_per_class": pc["support"],
        "true_counts": true_counts.tolist(),
        "pred_counts": pred_counts.tolist(),
        "n_total_frames": n_total,
        "n0": n0,
        "n1": n1,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


In [ ]:
# ===== 12) Train driver + LIVE plots =====
def plot_history_fig(history: List[Dict[str, float]]):
    if plt is None:
        return None
    df = pd.DataFrame(history)
    if len(df) == 0:
        fig = plt.figure(figsize=(10, 3))
        plt.title("No history yet")
        plt.axis("off")
        return fig

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Loss
    ax = axes[0]
    if "train/loss" in df.columns: ax.plot(df["epoch"], df["train/loss"], label="train/loss")
    if "val/loss" in df.columns:   ax.plot(df["epoch"], df["val/loss"],   label="val/loss")
    ax.set_title("Loss")
    ax.set_xlabel("epoch")
    ax.grid(True)
    ax.legend()

    # Macro-F1
    ax = axes[1]
    if "val/macro_f1" in df.columns:
        ax.plot(df["epoch"], df["val/macro_f1"], label="val/macro_f1")
    ax.set_title("Macro-F1")
    ax.set_xlabel("epoch")
    ax.grid(True)
    ax.legend()

    fig.tight_layout()
    return fig

def train_model(
    model,
    train_loader,
    val_loader,
    epochs: int,
    run_dir: Path,
    class_weights: torch.Tensor,
    optimizer,
    scheduler=None,
    scheduler_step: str = "epoch",     # "iter"|"epoch"|"plateau"
    label_smoothing: float = 0.05,
    amp: bool = True,
    grad_clip: float = 0.0,
    grad_accum_steps: int = 1,
    loss_type: str = "ce",
    focal_gamma: float = 2.0,
):
    run_dir.mkdir(parents=True, exist_ok=True)

    scaler = torch.amp.GradScaler("cuda", enabled=(amp and device == "cuda"))

    best_f1 = -1.0
    history: List[Dict[str, float]] = []

    tb_writer = None
    if bool(CFG.get("use_tensorboard", True)):
        try:
            from torch.utils.tensorboard import SummaryWriter
            tb_dir = run_dir / "tb"
            tb_writer = SummaryWriter(log_dir=str(tb_dir))
            print(f"[tb] logging to: {tb_dir}")
        except Exception as e:
            print(f"[tb][warn] TensorBoard disabled: {e}")

    log_handle  = display("", display_id=True)
    plot_handle = display("", display_id=True)

    # --- Boundary-Aware Loss ---
    boundary_loss_fn = None
    if bool(CFG.get("use_boundary_loss", False)):
        from models import BoundaryAwareLoss
        bw = float(CFG.get("boundary_weight", 2.0))
        br = int(CFG.get("boundary_radius", 3))
        boundary_loss_fn = BoundaryAwareLoss(boundary_weight=bw, boundary_radius=br).to(device)
        print(f"[loss] BoundaryAwareLoss enabled: weight={bw}, radius={br}")

    # --- Flow freeze support ---
    _freeze_flow_ep = int(CFG.get("freeze_flow_epochs", 0))
    _flow_frozen = False
    if _freeze_flow_ep > 0 and hasattr(model, 'flow_encoder'):
        for p in model.flow_encoder.parameters():
            p.requires_grad = False
        _flow_frozen = True
        print(f"[freeze] flow_encoder frozen for first {_freeze_flow_ep} epochs")

    for epoch in range(epochs):
        t0 = time.time()

        # Unfreeze flow encoder after N epochs
        if _flow_frozen and epoch >= _freeze_flow_ep:
            for p in model.flow_encoder.parameters():
                p.requires_grad = True
            _flow_frozen = False
            print(f"[freeze] flow_encoder UNFROZEN at epoch {epoch}")

        # ---- train
        tr = training_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            device=device,
            scaler=scaler if (amp and device == "cuda") else None,
            grad_clip=grad_clip,
            grad_accum_steps=grad_accum_steps,
            class_weights=class_weights,
            label_smoothing=label_smoothing,
            loss_type=loss_type,
            focal_gamma=focal_gamma,
            scheduler=scheduler,
            scheduler_step=scheduler_step,
            boundary_loss_fn=boundary_loss_fn,
        )

        # ---- val
        val = validation_epoch(
            model=model,
            loader=val_loader,
            device=device,
            num_classes=2,
            class_weights=class_weights,
            label_smoothing=label_smoothing,
            loss_type=loss_type,
            focal_gamma=focal_gamma,
        )

        print("F1 per class:", val["f1_per_class"])
        print("Precision per class:", val["prec_per_class"])
        print("Recall per class:", val["rec_per_class"])
        print("Support per class:", val["support_per_class"])
        print(f"VAL frames total={val['n_total_frames']}  n0={val['n0']}  n1={val['n1']}")

        # ---- LR bookkeeping (поддержка нескольких групп)
        lrs_before_sched = [pg["lr"] for pg in optimizer.param_groups]  # это LR ПОСЛЕ train_epoch (и после iter-step если он был)
        lr_train_end = tr.get("lr", lrs_before_sched[0])

        # ---- scheduler step per-EPOCH / PLATEAU (если надо)
        if scheduler is not None:
            if scheduler_step == "epoch":
                scheduler.step()
            elif scheduler_step == "plateau":
                scheduler.step(val["loss"])
            # если "iter" — уже шагали внутри training_epoch

        lrs_after_sched = [pg["lr"] for pg in optimizer.param_groups]

        row = {
            "epoch": epoch,
            "time_sec": float(time.time() - t0),

            "train/loss": float(tr["loss"]),
            "train/lr_end": float(lr_train_end),  # LR конца эпохи (как есть)
            "train/lrs_end": [float(x) for x in lrs_before_sched],

            "val/loss": float(val["loss"]),
            "val/macro_f1": float(val["macro_f1"]),

            "val/pred_dist_0": float(val["pred_dist"][0]),
            "val/pred_dist_1": float(val["pred_dist"][1]),
            "val/true_dist_0": float(val["true_dist"][0]),
            "val/true_dist_1": float(val["true_dist"][1]),

            # полезно, чтобы видеть что scheduler сделал (для epoch/plateau)
            "sched/lrs_after": [float(x) for x in lrs_after_sched],
        }
        history.append(row)

        (run_dir / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")

        if tb_writer is not None:
            tb_writer.add_scalar("train/loss", row["train/loss"], epoch)
            tb_writer.add_scalar("train/lr_end", row["train/lr_end"], epoch)
            tb_writer.add_scalar("val/loss", row["val/loss"], epoch)
            tb_writer.add_scalar("val/macro_f1", row["val/macro_f1"], epoch)
            if "f1_per_class" in val and len(val["f1_per_class"]) >= 2:
                tb_writer.add_scalar("val/f1_non_violence", float(val["f1_per_class"][0]), epoch)
                tb_writer.add_scalar("val/f1_violence", float(val["f1_per_class"][1]), epoch)
            tb_writer.flush()

        # ---- обновляем best ДО сохранения last.pt, чтобы last всегда содержал актуальный best_f1
        improved = row["val/macro_f1"] > best_f1
        if improved:
            best_f1 = row["val/macro_f1"]

        model_to_save = model.module if hasattr(model, "module") else model
        ckpt = {
            "model": model_to_save.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict() if scheduler is not None else None,
            "scaler": scaler.state_dict() if (amp and device == "cuda") else None,
            "epoch": epoch,
            "best_f1": best_f1,
            "scheduler_step": scheduler_step,
        }
        torch.save(ckpt, run_dir / "last.pt")
        if improved:
            torch.save(ckpt, run_dir / "best.pt")

        # ---- logging
        # (покажем LR конца эпохи и LR после scheduler.step(), если он менялся)
        lr_show = lrs_before_sched[0]
        lr_after_show = lrs_after_sched[0]
        log_text = (
            f"epoch={epoch}  time={row['time_sec']:.1f}s  "
            f"lr_end={lr_show:.6g}  lr_after_sched={lr_after_show:.6g}\n"
            f"train/loss={row['train/loss']:.4f}  val/loss={row['val/loss']:.4f}  "
            f"val/macro_f1={row['val/macro_f1']:.4f}  best_f1={best_f1:.4f}\n"
            f"pred_dist={val['pred_dist']}  true_dist={val['true_dist']}\n"
            f"scheduler_step={scheduler_step}  run_dir={run_dir}"
        )
        log_handle.update(log_text)

        fig = plot_history_fig(history)
        if fig is not None:
            plot_handle.update(fig)
            plt.close(fig)

    if tb_writer is not None:
        tb_writer.close()

    print("BEST val/macro_f1:", best_f1)
    print("run_dir:", run_dir)
    return history, best_f1, run_dir




In [ ]:
# ===== model =====
# Меняй CFG["backbone"] выше чтобы переключаться между моделями
backbone = CFG["backbone"]
print(f"Building model: {backbone}")

model = build_model(
    backbone=backbone,
    num_classes=2,
    pretrained=True,
    dropout=0.3,
    cfg=CFG,
).to(device)

if bool(CFG.get("use_dataparallel", False)) and device == "cuda":
    req_ids = CFG.get("data_parallel_device_ids", [0, 1])
    if not isinstance(req_ids, (list, tuple)):
        req_ids = [int(req_ids)]
    req_ids = [int(x) for x in req_ids]
    n_gpu = torch.cuda.device_count()
    valid_ids = [i for i in req_ids if 0 <= i < n_gpu]
    if len(valid_ids) >= 2:
        model = torch.nn.DataParallel(model, device_ids=valid_ids, output_device=valid_ids[0])
        print(f"[multi-gpu] DataParallel enabled on device_ids={valid_ids}")
    else:
        print(f"[multi-gpu][warn] requested {req_ids}, available cuda count={n_gpu}. Running single-GPU.")

# Считаем параметры
_raw_model = model.module if hasattr(model, "module") else model
total_params = sum(p.numel() for p in _raw_model.parameters())
trainable_params = sum(p.numel() for p in _raw_model.parameters() if p.requires_grad)
print(f"Total params:     {total_params/1e6:.1f}M")
print(f"Trainable params: {trainable_params/1e6:.1f}M")

# --- memory saver: gradient checkpointing on timm encoders ---
if bool(CFG.get("encoder_grad_checkpointing", True)):
    _enc_owner = model.module if hasattr(model, "module") else model
    _gc_done = []
    for _name in ["encoder", "rgb_encoder", "flow_encoder"]:
        _enc = getattr(_enc_owner, _name, None)
        if _enc is not None and hasattr(_enc, "set_grad_checkpointing"):
            try:
                _enc.set_grad_checkpointing(True)
                _gc_done.append(_name)
            except Exception as _e:
                print(f"[gc][warn] {_name}: {_e}")
    if _gc_done:
        print(f"[gc] enabled for: {_gc_done}")


def _resolve_rgb_target_encoder(m):
    if hasattr(m, 'rgb_encoder'):
        return m.rgb_encoder, 'rgb_encoder'
    if hasattr(m, 'encoder') and bool(CFG.get('use_skeleton', False)):
        return m.encoder, 'encoder'
    return None, None

_target_encoder, _target_encoder_name = _resolve_rgb_target_encoder(_raw_model)

# --- Load pretrained RGB backbone from checkpoint ---
_rgb_ckpt_path = CFG.get('rgb_backbone_checkpoint', None)
if _rgb_ckpt_path is not None:
    if _target_encoder is None:
        print('[warn] rgb_backbone_checkpoint is set, but no RGB encoder found on model')
    else:
        _p = Path(_rgb_ckpt_path)
        if not _p.is_absolute():
            p_local = Path.cwd() / _p
            p_proj = Path('') / _p
            _p = p_local if p_local.exists() else p_proj
        if _p.is_dir():
            _p = _p / 'best.pt'

        if _p.exists():
            _ckpt = torch.load(_p, map_location=device)
            _state = _ckpt['model'] if isinstance(_ckpt, dict) and 'model' in _ckpt else _ckpt

            _raw = {}
            for k, v in _state.items():
                if k.startswith('encoder.'):
                    _raw[k.replace('encoder.', '', 1)] = v
                elif k.startswith('rgb_encoder.'):
                    _raw[k.replace('rgb_encoder.', '', 1)] = v

            if _raw:
                missing, unexpected = _target_encoder.load_state_dict(_raw, strict=False)
                print(f"[pretrained] RGB backbone loaded into '{_target_encoder_name}' from {_p}")
                print(f"  loaded: {len(_raw)} params, missing: {len(missing)}, unexpected: {len(unexpected)}")
            else:
                print('[warn] No encoder.* or rgb_encoder.* keys found in checkpoint')
        else:
            print(f"[warn] RGB checkpoint not found: {_p}")

# --- Freeze RGB backbone if configured ---
if bool(CFG.get('freeze_rgb_backbone', False)):
    if _target_encoder is None:
        print('[warn] freeze_rgb_backbone=True, but no RGB encoder found')
    else:
        for p in _target_encoder.parameters():
            p.requires_grad = False
        _frozen = sum(p.numel() for p in _target_encoder.parameters())
        print(f"[freeze] {_target_encoder_name} frozen ({_frozen/1e6:.1f}M params)")

# Быстрая проверка forward pass
with torch.no_grad():
    _dummy = torch.zeros(2, 3, CFG['clip_len'], CFG['img_size'], CFG['img_size']).to(device)

if bool(CFG.get('use_skeleton', False)):
    _dummy_skel = torch.zeros(2, CFG['clip_len'], int(CFG.get('skeleton_dim', 406))).to(device)
    _out = model(_dummy, skeleton=_dummy_skel)
elif bool(CFG.get('use_flow', False)):
    _dummy_flow = torch.zeros(2, 3, CFG['clip_len'], CFG['img_size'], CFG['img_size']).to(device)
    _out = model(_dummy, flow=_dummy_flow)
else:
    _out = model(_dummy)

print(f"Forward check: input {tuple(_dummy.shape)} -> output {tuple(_out.shape)}")
assert _out.shape == (2, 2, CFG['clip_len']), f"Unexpected output shape: {_out.shape}"
print('Forward pass OK')




In [ ]:
# ===== optimizer =====
# Дифференциальный LR: backbone обучается медленнее, голова быстрее.
# Это особенно важно для претренированных Video Swin / VideoMAE.

backbone_name = CFG["backbone"]


def get_param_groups(model, backbone_lr_mult: float = 0.1, weight_decay: float = 5e-4):
    """
    backbone_lr_mult: насколько backbone медленнее головы.
    0.1 -> backbone LR в 10x меньше чем у головы.
    """
    head_params_decay, head_params_no_decay = [], []
    backbone_params_decay, backbone_params_no_decay = [], []

    head_names = {"head", "lstm", "drop"}

    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        is_head = any(h in name for h in head_names)
        no_wd = (p.ndim == 1 or name.endswith(".bias"))
        if is_head:
            (head_params_no_decay if no_wd else head_params_decay).append(p)
        else:
            (backbone_params_no_decay if no_wd else backbone_params_decay).append(p)

    return [
        {"params": backbone_params_decay,    "weight_decay": weight_decay, "lr_mult": backbone_lr_mult},
        {"params": backbone_params_no_decay, "weight_decay": 0.0,          "lr_mult": backbone_lr_mult},
        {"params": head_params_decay,        "weight_decay": weight_decay, "lr_mult": 1.0},
        {"params": head_params_no_decay,     "weight_decay": 0.0,          "lr_mult": 1.0},
    ]

if backbone_name == "resnet18":
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim == 1 or name.endswith(".bias"):
            no_decay.append(p)
        else:
            decay.append(p)

    resnet_lr = float(CFG.get("resnet_lr", 1e-3))
    resnet_wd = float(CFG.get("resnet_weight_decay", 5e-4))

    param_groups = [
        {"params": decay,    "weight_decay": resnet_wd, "lr": resnet_lr},
        {"params": no_decay, "weight_decay": 0.0,       "lr": resnet_lr},
    ]
    optimizer = torch.optim.SGD(
        param_groups,
        momentum=float(CFG.get("resnet_momentum", 0.9)),
        nesterov=bool(CFG.get("resnet_nesterov", True)),
    )
else:
    base_lr = float(CFG.get("base_lr", 1e-4))
    backbone_lr_mult = float(CFG.get("backbone_lr_mult", 0.1))
    adamw_weight_decay = float(CFG.get("adamw_weight_decay", 5e-4))

    pg = get_param_groups(
        model,
        backbone_lr_mult=backbone_lr_mult,
        weight_decay=adamw_weight_decay,
    )
    for g in pg:
        g["lr"] = base_lr * g.pop("lr_mult")
    optimizer = torch.optim.AdamW(pg, lr=base_lr)

print("Optimizer:", type(optimizer).__name__)
for i, g in enumerate(optimizer.param_groups):
    print(f"  group {i}: lr={g['lr']:.2e}  wd={g['weight_decay']}  n_params={len(g['params'])}")



In [ ]:
# ===== scheduler =====
backbone_name = CFG["backbone"]
steps_per_epoch = math.ceil(len(train_loader) / CFG["grad_accum_steps"])

if backbone_name == "resnet18":
    max_lr = float(CFG.get("resnet_max_lr", 0.05))
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr,
        epochs=CFG["epochs"],
        steps_per_epoch=steps_per_epoch,
        pct_start=float(CFG.get("resnet_pct_start", 0.1)),
        anneal_strategy="cos",
        cycle_momentum=True,
        base_momentum=float(CFG.get("resnet_base_momentum", 0.85)),
        max_momentum=float(CFG.get("resnet_max_momentum", 0.95)),
        div_factor=float(CFG.get("resnet_div_factor", 25.0)),
        final_div_factor=float(CFG.get("resnet_final_div_factor", 1e4)),
    )
    scheduler_step_mode = "iter"
else:
    peak_mult = float(CFG.get("onecycle_peak_mult", 10.0))
    max_lrs = [g["lr"] * peak_mult for g in optimizer.param_groups]

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lrs,
        epochs=CFG["epochs"],
        steps_per_epoch=steps_per_epoch,
        pct_start=float(CFG.get("onecycle_pct_start", 0.15)),
        anneal_strategy="cos",
        cycle_momentum=False,
        div_factor=float(CFG.get("onecycle_div_factor", 25.0)),
        final_div_factor=float(CFG.get("onecycle_final_div", 1e4)),
    )
    scheduler_step_mode = "iter"

print(f"Scheduler: OneCycleLR, step_mode={scheduler_step_mode}")
print(f"Steps per epoch: {steps_per_epoch}, total steps: {steps_per_epoch * CFG['epochs']}")



In [ ]:
!nvidia-smi

In [ ]:
# ===== actual training =====
print("CFG backend:", CFG["backend"])
print("train_loader backend:", train_loader.dataset.backend, "frames_root:", train_loader.dataset.frames_root)
print("val_loader backend:", val_loader.dataset.backend, "frames_root:", val_loader.dataset.frames_root)
assert CFG["backend"] == "frames", f"Expected CFG backend=frames, got {CFG['backend']}"
assert train_loader.dataset.backend == "frames", f"Train loader backend is {train_loader.dataset.backend}"
assert val_loader.dataset.backend == "frames", f"Val loader backend is {val_loader.dataset.backend}"
assert Path(train_loader.dataset.frames_root).exists(), f"Missing train frames root: {train_loader.dataset.frames_root}"
assert Path(val_loader.dataset.frames_root).exists(), f"Missing val frames root: {val_loader.dataset.frames_root}"

run_dir = Path(CFG["run_dir"]) / f"{CFG['backbone']}_{time.strftime('%Y%m%d_%H%M%S')}"
run_dir.mkdir(parents=True, exist_ok=True)

# ===== RUN METADATA / REPRO SNAPSHOT =====
import hashlib, socket, getpass, platform, copy

def _sha256(path: Path):
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

effective_cfg = copy.deepcopy(CFG)
manifest = {
    "created_at": time.strftime('%Y-%m-%d %H:%M:%S'),
    "run_dir": str(run_dir),
    "train_preset": CFG.get("train_preset"),
    "backbone": CFG.get("backbone"),
    "hostname": socket.gethostname(),
    "user": getpass.getuser(),
    "python": platform.python_version(),
    "device": device,
    "torch_cuda_visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
    "files": {
        "notebook": "exp_backbones_v2.ipynb",
        "models_py": "models.py",
        "presets": str(presets_path),
    },
    "sha256": {
        "models_py": _sha256(Path('models.py')),
        "presets": _sha256(Path(presets_path)),
    },
    "cfg": effective_cfg,
}

(run_dir / "cfg_effective.json").write_text(json.dumps(effective_cfg, ensure_ascii=False, indent=2), encoding="utf-8")
(run_dir / "run_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
Path('LATEST_RUN.txt').write_text(str(run_dir) + '\n', encoding='utf-8')

ledger_path = Path('EXPERIMENT_LEDGER.jsonl')
ledger_row = {
    "created_at": manifest["created_at"],
    "run_dir": str(run_dir),
    "train_preset": CFG.get("train_preset"),
    "backbone": CFG.get("backbone"),
    "clip_len": CFG.get("clip_len"),
    "frame_step": CFG.get("frame_step"),
    "train_batch_size": CFG.get("train_batch_size"),
    "grad_accum_steps": CFG.get("grad_accum_steps"),
    "base_lr": CFG.get("base_lr"),
    "use_dataparallel": CFG.get("use_dataparallel"),
}
with ledger_path.open('a', encoding='utf-8') as f:
    f.write(json.dumps(ledger_row, ensure_ascii=False) + '\n')

print("[repro] saved:", run_dir / "cfg_effective.json")
print("[repro] saved:", run_dir / "run_manifest.json")
print("[repro] latest run pointer:", 'LATEST_RUN.txt')

history, best_f1, run_dir = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=CFG["epochs"],
    run_dir=run_dir,
    class_weights=class_weights,
    optimizer=optimizer,
    scheduler=scheduler,
    scheduler_step=scheduler_step_mode,
    label_smoothing=CFG["label_smoothing"],
    loss_type=CFG.get("loss_type", "ce"),
    focal_gamma=float(CFG.get("focal_gamma", 2.0)),
    amp=CFG["amp"],
    grad_clip=CFG["grad_clip"],
    grad_accum_steps=CFG["grad_accum_steps"],
)

tb_dir = run_dir / "tb"
if tb_dir.exists():
    print(f"[tb] logdir: {tb_dir}")
    print(f"[tb] launch: tensorboard --logdir {tb_dir} --port 6006 --host 0.0.0.0")

if CFG.get("auto_load_best_after_train", True):
    best_ckpt = run_dir / "best.pt"
    if best_ckpt.exists():
        ckpt = torch.load(best_ckpt, map_location=device)
        model_for_load = model.module if hasattr(model, "module") else model
        model_for_load.load_state_dict(ckpt["model"], strict=True)
        model.to(device)
        model.eval()
        print(f"Auto-loaded best checkpoint: {best_ckpt} (best_f1={ckpt.get('best_f1')})")
    else:
        print(f"[warn] best checkpoint not found: {best_ckpt}")







In [ ]:
!nvidia-smi

In [ ]:
# ===== 13) Inference: per-frame labels for a whole video (sliding window) =====
@torch.no_grad()
def infer_video_per_frame(
    model: nn.Module,
    video_path: Optional[Path] = None,
    video_id: Optional[str] = None,
    frames_root: Optional[Path] = None,
    ann_path: Optional[Path] = None,
    clip_len: int = 16,
    frame_step: int = 1,
    stride: Optional[int] = None,
    threshold: float = 0.5,
    transform=None,
    backend: str = "pyav",
    device: str = "cuda",
    debug: bool = True,
    flow_frames_root: Optional[Path] = None,
    skeleton_root: Optional[Path] = None,
):
    model.eval()
    if transform is None:
        transform = val_tfms
    if stride is None:
        stride = max(1, (clip_len * frame_step) // 2)

    n_frames = 0
    if ann_path is not None and Path(ann_path).exists():
        out = load_frame_labels(Path(ann_path))
        if isinstance(out, tuple):
            labels = out[0]
            frame_numbers = out[1] if len(out) > 1 else None
        else:
            labels = out
            frame_numbers = None
        if frame_numbers is not None and len(frame_numbers) > 0:
            n_frames = int(np.max(frame_numbers))
        else:
            n_frames = int(len(labels))

    frame_index = None
    frames_dir = None
    if backend == "frames":
        if video_id is None:
            if video_path is None:
                raise ValueError("For backend='frames' provide video_id or video_path")
            video_id = Path(video_path).stem
        if frames_root is None:
            raise ValueError("For backend='frames' provide frames_root")
        frames_dir = Path(frames_root) / str(video_id)
        frame_index = _build_frame_index(frames_dir)
        if n_frames <= 0 and frame_index is not None and len(frame_index[0]) > 0:
            n_frames = int(frame_index[0][-1])

    if n_frames <= 0 and backend != "frames" and video_path is not None:
        try:
            import cv2
            cap = cv2.VideoCapture(str(video_path))
            if cap.isOpened():
                n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
            cap.release()
        except Exception:
            pass

    if n_frames <= 0 and backend != "frames" and video_path is not None:
        try:
            import av
            container = av.open(str(video_path))
            stream = next(s for s in container.streams if s.type == "video")
            n_frames = int(stream.frames) if stream.frames else 0
            if n_frames <= 0 and stream.duration and stream.time_base and stream.average_rate:
                n_frames = int(float(stream.duration * stream.time_base) * float(stream.average_rate))
            container.close()
        except Exception:
            n_frames = 0

    if debug:
        src = str(frames_dir) if backend == "frames" else (video_path.name if video_path is not None else "None")
        print(f"[infer] src={src} n_frames={n_frames} backend={backend} frame_step={frame_step} stride={stride}")

    if n_frames <= 0:
        return np.zeros((0,), np.float32), np.zeros((0,), np.int64)

    prob_sum = np.zeros((n_frames,), dtype=np.float32)
    prob_cnt = np.zeros((n_frames,), dtype=np.float32)

    span = (clip_len - 1) * frame_step + 1
    last_start = max(0, n_frames - span)
    starts = list(range(0, last_start + 1, stride))
    if len(starts) == 0:
        starts = [0]
    if starts[-1] != last_start:
        starts.append(last_start)

    skel_all = None
    if skeleton_root is not None and video_id is not None:
        skel_path = Path(skeleton_root) / f"{video_id}.npy"
        if skel_path.exists():
            try:
                skel_all = np.load(skel_path)
            except Exception:
                skel_all = None

    for start in tqdm(starts, desc="infer", dynamic_ncols=True, disable=True):
        if backend == "frames":
            rgb = read_clip_rgb(
                video_path=None,
                start=start,
                clip_len=clip_len,
                frame_step=frame_step,
                backend=backend,
                frames_dir=frames_dir,
                frame_index=frame_index,
            )
        else:
            rgb = read_clip_rgb(
                video_path=video_path,
                start=start,
                clip_len=clip_len,
                frame_step=frame_step,
                backend=backend,
            )

        if rgb is None or rgb.shape[0] != clip_len:
            rgb = np.stack([dummy_frame(CFG["img_size"])] * clip_len, axis=0)

        frames = []
        for t in range(clip_len):
            fr_t = torch.from_numpy(rgb[t]).permute(2, 0, 1)
            frames.append(transform(fr_t))
        clip = torch.stack(frames, dim=0).permute(1, 0, 2, 3).unsqueeze(0).to(device)

        idxs = start + np.arange(clip_len) * frame_step
        idxs = np.clip(idxs, 0, n_frames - 1).astype(int)

        if flow_frames_root is not None:
            flow_dir = Path(flow_frames_root) / str(video_id)
            flow_fi = _build_frame_index(flow_dir) if flow_dir.exists() else None
            flow_rgb = read_clip_rgb(
                video_path=None,
                start=start,
                clip_len=clip_len,
                frame_step=frame_step,
                backend="frames",
                frames_dir=flow_dir,
                frame_index=flow_fi,
            )
            if flow_rgb is None or flow_rgb.shape[0] != clip_len:
                flow_rgb = np.stack([dummy_frame(CFG["img_size"])] * clip_len, axis=0)

            flow_frames_list = []
            for t in range(clip_len):
                fr_t = torch.from_numpy(flow_rgb[t]).permute(2, 0, 1)
                flow_frames_list.append(transform(fr_t))
            flow_clip = torch.stack(flow_frames_list, dim=0).permute(1, 0, 2, 3).unsqueeze(0).to(device)

            logits = model(clip, flow=flow_clip)

        elif skeleton_root is not None:
            feat_dim = 406
            if skel_all is None or not isinstance(skel_all, np.ndarray) or skel_all.ndim != 2:
                skel_clip = np.zeros((clip_len, feat_dim), dtype=np.float32)
            else:
                feat_dim = int(skel_all.shape[1]) if skel_all.shape[1] > 0 else 406
                safe_idxs = np.clip(idxs, 0, len(skel_all) - 1).astype(int) if len(skel_all) > 0 else None
                if safe_idxs is None:
                    skel_clip = np.zeros((clip_len, feat_dim), dtype=np.float32)
                else:
                    skel_clip = skel_all[safe_idxs].astype(np.float32)

            skeleton = torch.from_numpy(skel_clip).unsqueeze(0).to(device)
            logits = model(clip, skeleton=skeleton)

        else:
            logits = model(clip)

        probs1 = torch.softmax(logits, dim=1)[:, 1, :].squeeze(0).detach().cpu().numpy()

        for k, fi in enumerate(idxs):
            prob_sum[fi] += float(probs1[k])
            prob_cnt[fi] += 1.0

    probs = prob_sum / np.maximum(prob_cnt, 1.0)
    pred = (probs >= threshold).astype(np.int64)

    if debug:
        covered = int((prob_cnt > 0).sum())
        print(f"[infer] covered_frames={covered}/{n_frames} ({covered/n_frames:.1%}) probs[min,max]=({probs.min():.3f},{probs.max():.3f})")

    return probs, pred

def smooth_probs(probs: np.ndarray, win: int = 7) -> np.ndarray:
    if win <= 1 or probs.size == 0:
        return probs
    pad = win // 2
    x = np.pad(probs, (pad, pad), mode="edge")
    kernel = np.ones((win,), dtype=np.float32) / win
    return np.convolve(x, kernel, mode="valid")



In [ ]:
ckpt = torch.load(run_dir / "best.pt", map_location=device)  # или Path(".../best.pt")
model.load_state_dict(ckpt["model"])
model.to(device)
model.eval()
print("loaded best.pt, best_f1 =", ckpt.get("best_f1"))



In [ ]:
# ===== FULL VAL (ALL VIDEOS) EVAL CELL =====
import torch
import numpy as np
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import f1_score, confusion_matrix, classification_report

@torch.no_grad()
def eval_full_val_videos(
    model,
    val_meta,
    videos_root: Optional[Path] = None,
    frames_root: Optional[Path] = None,
    video_ext: str = ".mp4",
    threshold: float = 0.5,
    smooth_win: int = 1,
    clip_len: int = 16,
    frame_step: int = 1,
    stride: Optional[int] = None,
    transform=None,
    backend: str = "pyav",
    device: str = "cuda",
    verbose: bool = True,
    flow_frames_root: Optional[Path] = None,
    skeleton_root: Optional[Path] = None,
):
    model.eval()
    model.to(device)

    y_true_all = []
    y_pred_all = []
    per_video = []

    if "video" not in val_meta.columns:
        raise ValueError("val_meta must contain column 'video'.")
    if "csv_path" not in val_meta.columns:
        raise ValueError("val_meta must contain column 'csv_path'.")

    for i in tqdm(range(len(val_meta)), desc="Full-val over videos"):
        row = val_meta.iloc[i]

        video_id = str(row["video"])
        ann_path = Path(row["csv_path"])
        video_path = None

        if backend == "frames":
            if frames_root is None:
                raise ValueError("backend='frames' requires frames_root")
            frames_dir = Path(frames_root) / video_id
            if not frames_dir.exists():
                if verbose:
                    print(f"[skip] frames dir not found: {frames_dir}")
                continue
        else:
            if videos_root is None:
                raise ValueError(f"backend='{backend}' requires videos_root")
            video_path = Path(videos_root) / (video_id + video_ext)
            if not video_path.exists():
                if verbose:
                    print(f"[skip] video not found: {video_path}")
                continue

        if not ann_path.exists():
            if verbose:
                print(f"[skip] ann not found: {ann_path}")
            continue

        out = load_frame_labels(ann_path)
        y_full = out[0] if isinstance(out, tuple) else out
        y_full = np.asarray(y_full)

        probs, _pred_unused = infer_video_per_frame(
            model=model,
            video_path=video_path,
            video_id=video_id,
            frames_root=frames_root,
            ann_path=ann_path,
            clip_len=clip_len,
            frame_step=frame_step,
            stride=stride,
            threshold=threshold,
            transform=transform,
            backend=backend,
            device=device,
            debug=False,
            flow_frames_root=flow_frames_root,
            skeleton_root=skeleton_root,
        )
        probs = np.asarray(probs)
        if smooth_win is not None and int(smooth_win) > 1:
            probs = smooth_probs(probs, win=int(smooth_win))
        pred = (probs >= float(threshold)).astype(np.int64)

        L = min(len(y_full), len(pred))
        y_full = y_full[:L]
        pred = pred[:L]

        m = (y_full != -1)
        yt = y_full[m].astype(np.int64)
        yp = pred[m].astype(np.int64)

        if yt.size == 0:
            if verbose:
                print(f"[skip] no labeled frames after mask: {video_id}")
            continue

        y_true_all.append(yt)
        y_pred_all.append(yp)

        pv_f1 = f1_score(yt, yp, average="macro", labels=[0, 1])
        per_video.append((video_id, float(pv_f1), int((yt == 1).sum()), int(len(yt))))

    if len(y_true_all) == 0:
        raise RuntimeError("No labeled frames were evaluated. Check paths/annotations/backend.")

    y_true_all = np.concatenate(y_true_all)
    y_pred_all = np.concatenate(y_pred_all)

    macro_f1 = f1_score(y_true_all, y_pred_all, average="macro", labels=[0, 1])
    f1_per_class = f1_score(y_true_all, y_pred_all, average=None, labels=[0, 1])
    cm = confusion_matrix(y_true_all, y_pred_all, labels=[0, 1])

    report = classification_report(
        y_true_all,
        y_pred_all,
        labels=[0, 1],
        target_names=["non-violence (0)", "violence (1)"],
        digits=4,
        zero_division=0,
    )

    per_video_sorted = sorted(per_video, key=lambda x: x[1])

    return {
        "macro_f1": float(macro_f1),
        "f1_non_violence": float(f1_per_class[0]),
        "f1_violence": float(f1_per_class[1]),
        "confusion_matrix_01": cm,
        "classification_report": report,
        "per_video_sorted": per_video_sorted,
        "n_frames_eval": int(len(y_true_all)),
        "pos_frames_eval": int((y_true_all == 1).sum()),
        "neg_frames_eval": int((y_true_all == 0).sum()),
        "threshold": float(threshold),
        "clip_len": int(clip_len),
        "frame_step": int(frame_step),
        "stride": None if stride is None else int(stride),
        "backend": backend,
        "smooth_win": int(smooth_win),
    }




In [ ]:
res = eval_full_val_videos(
    model=model,
    val_meta=val_meta,
    frames_root=Path(VAL_FRAMES),
    threshold=0.5,
    smooth_win=1,
    clip_len=CFG["clip_len"],
    frame_step=1,
    transform=val_tfms,
    backend="frames",
    device=device,
    flow_frames_root=VAL_FLOW_FRAMES if USE_FLOW else None,
    skeleton_root=VAL_SKELETON if USE_SKELETON else None,
)

print('=== FULL VAL RESULTS (stable) ===')
print(f"macro_f1: {res['macro_f1']:.4f}")
print(f"f1_violence: {res['f1_violence']:.4f}")
print(f"f1_non_violence: {res['f1_non_violence']:.4f}")

# === AUTO-SAVE full-val results ===
_fullval_log = {
    "backbone": CFG.get("backbone", "unknown"),
    "run_dir": str(run_dir) if "run_dir" in dir() else "unknown",
    "macro_f1": float(res["macro_f1"]),
    "f1_violence": float(res["f1_violence"]),
    "f1_non_violence": float(res["f1_non_violence"]),
    "threshold": float(res.get("threshold", 0.5)),
    "smooth_win": int(res.get("smooth_win", 1)),
    "clip_len": int(res.get("clip_len", CFG.get("clip_len", 16))),
    "frame_step": int(res.get("frame_step", 1)),
    "n_frames_eval": int(res.get("n_frames_eval", 0)),
    "pos_frames_eval": int(res.get("pos_frames_eval", 0)),
    "neg_frames_eval": int(res.get("neg_frames_eval", 0)),
    "classification_report": str(res.get("classification_report", "")),
    "timestamp": time.strftime("%Y%m%d_%H%M%S"),
    "cfg_snapshot": {k: str(v) if isinstance(v, Path) else v for k, v in CFG.items() if not callable(v)},
}

# Сохраняем рядом с run_dir
if "run_dir" in dir() and run_dir is not None:
    _fv_path = Path(run_dir) / "full_val_result.json"
    try:
        _fv_path.write_text(json.dumps(_fullval_log, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
        print(f"[saved] {_fv_path}")
    except Exception as _e:
        print(f"[warn] could not save to run_dir: {_e}")

# Также в centralized лог (append-mode)
_fv_central = Path(CFG.get("root", ".")) / ".." / ".." / "full_val_log.jsonl"
try:
    _fv_central = Path("full_val_logs") / "all_results.jsonl"
    _fv_central.parent.mkdir(parents=True, exist_ok=True)
    with open(_fv_central, "a", encoding="utf-8") as _f:
        _f.write(json.dumps(_fullval_log, ensure_ascii=False, default=str) + "\n")
    print(f"[saved] appended to {_fv_central}")
except Exception as _e:
    print(f"[warn] could not append to central log: {_e}")


In [ ]:
# ===== 15) Threshold sweep on cached FULL-VAL probabilities =====
from sklearn.metrics import f1_score

@torch.no_grad()
def build_full_val_cache(
    model,
    val_meta,
    frames_root: Path,
    transform,
    backend: str = "frames",
    clip_len: int = 16,
    frame_step: int = 1,
    stride: Optional[int] = None,
    device: str = "cuda",
    flow_frames_root: Optional[Path] = None,
    skeleton_root: Optional[Path] = None,
):
    """
    Runs full-video inference once and stores (y_true, probs) per video.
    Later you can sweep thresholds instantly without re-running model inference.
    """
    cache = []
    for i in tqdm(range(len(val_meta)), desc="cache full-val"):
        row = val_meta.iloc[i]
        video_id = str(row["video"])
        ann_path = Path(row["csv_path"])

        probs, _ = infer_video_per_frame(
            model=model,
            video_path=None,
            video_id=video_id,
            frames_root=frames_root,
            ann_path=ann_path,
            clip_len=clip_len,
            frame_step=frame_step,
            stride=stride,
            threshold=0.5,
            transform=transform,
            backend=backend,
            device=device,
            debug=False,
            flow_frames_root=flow_frames_root,
            skeleton_root=skeleton_root,
        )

        y_full = load_frame_labels(ann_path)
        y_full = y_full[0] if isinstance(y_full, tuple) else y_full
        y_full = np.asarray(y_full)

        L = min(len(y_full), len(probs))
        y = y_full[:L]
        p = np.asarray(probs[:L], dtype=np.float32)

        m = (y != -1)
        yt = y[m].astype(np.int64)
        pb = p[m].astype(np.float32)

        cache.append({
            "video": video_id,
            "y_true": yt,
            "probs": pb,
            "pos_true": int((yt == 1).sum()),
            "n": int(len(yt)),
        })
    return cache


def _smooth_1d(x: np.ndarray, win: int) -> np.ndarray:
    if win <= 1:
        return x
    pad = win // 2
    z = np.pad(x, (pad, pad), mode="edge")
    k = np.ones((win,), dtype=np.float32) / float(win)
    return np.convolve(z, k, mode="valid")


def score_full_val_cache(cache, threshold: float = 0.5, smooth_win: int = 1):
    ys, ps = [], []
    for item in cache:
        yt = item["y_true"]
        pb = item["probs"]
        pb2 = _smooth_1d(pb, smooth_win)
        ys.append(yt)
        ps.append(pb2)

    y = np.concatenate(ys)
    p = np.concatenate(ps)
    yp = (p >= float(threshold)).astype(np.int64)

    f1_0, f1_1 = f1_score(y, yp, average=None, labels=[0, 1])
    macro = f1_score(y, yp, average="macro", labels=[0, 1])
    return {
        "macro_f1": float(macro),
        "f1_non_violence": float(f1_0),
        "f1_violence": float(f1_1),
        "pred_pos": int((yp == 1).sum()),
        "total": int(len(yp)),
    }


def sweep_thresholds(cache, thresholds, smooth_wins=(1, 5, 7)):
    rows = []
    for win in smooth_wins:
        for thr in thresholds:
            s = score_full_val_cache(cache, threshold=float(thr), smooth_win=int(win))
            rows.append({
                "macro_f1": s["macro_f1"],
                "f1_violence": s["f1_violence"],
                "f1_non_violence": s["f1_non_violence"],
                "threshold": float(thr),
                "smooth_win": int(win),
                "pred_pos": s["pred_pos"],
                "total": s["total"],
            })
    rows = sorted(rows, key=lambda x: x["macro_f1"], reverse=True)
    return rows

# ---- usage ----
# 1) Build cache once (can take several minutes):
# cache = build_full_val_cache(
#     model=model,
#     val_meta=val_meta,
#     frames_root=Path(VAL_FRAMES),
#     transform=val_tfms,
#     backend="frames",
#     clip_len=CFG.get("clip_len", 16),
#     frame_step=1,
#     stride=None,
#     device=device,
# )
#
# 2) Sweep thresholds quickly:
# thresholds = np.arange(0.20, 0.61, 0.02)
# rows = sweep_thresholds(cache, thresholds, smooth_wins=(1, 5, 7))
# pd.DataFrame(rows).head(15)

